# FPN+V3：简化特征干扰分析

本分析用于检验一个有限、可证伪的命题：镜面高光是否会产生超出同区域、
同能量随机噪声的深层特征扰动，并且这种扰动是否与预测不一致相关。

Notebook 完全独立，不导入其他 notebook、不训练模型，只读取该模型已有的
Raw 权重。FMM 严格保持论文流程：灰度 99.5% 分位掩膜、0.6 倍缩放、
`cv2.INPAINT_TELEA`、半径 10、JPEG 质量 95、最终尺寸 384x288。

三种配对输入均以同一张 FMM 图像为基准：

- `FMM`：论文的修复图像，作为近似无高光参考。
- `Highlight`：只在高光 mask 内重新放回 Raw 图像像素，隔离结构化高光成分。
- `Matched noise`：在同一个 mask 内加入随机噪声，并将实现后的 L2 能量严格匹配
  `Highlight - FMM`；噪声候选的逐通道均值和标准差来自真实高光残差。

该设计提供支持性证据而不是因果“证明”。只有当高光相对匹配噪声产生更大的
深层特征偏移，且偏移与预测不一致相关时，才适合保留“表征层面的结构化干扰”表述。


In [1]:
# ============================================================
# 1. 依赖、路径与可修改配置
# ============================================================
import gc
import json
import math
import os
import re
import zlib
from pathlib import Path

os.environ.setdefault("SM_FRAMEWORK", "tf.keras")

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from scipy.stats import rankdata, spearmanr, wilcoxon

import segmentation_models as sm
sm.set_framework("tf.keras")


# 如果原电脑上的项目位置发生变化，只修改这一项。
PROJECT_ROOT = Path("D:/WJY/paper1")

MODEL_NAME = "FPN+V3"
MODEL_ARCHITECTURE = "FPN"
MODEL_CONFIGS = {MODEL_NAME: {"architecture": MODEL_ARCHITECTURE}}
SAFE_MODEL_NAME = re.sub(r"[^A-Za-z0-9._+-]+", "_", MODEL_NAME)
TEST_DATASETS = (
    "CVC-300",
    "CVC-ClinicDB",
    "CVC-ColonDB",
    "ETIS-LaribPolypDB",
    "Kvasir",
)

IMAGE_HEIGHT = 288
IMAGE_WIDTH = 384
IMAGE_SIZE = (IMAGE_WIDTH, IMAGE_HEIGHT)
PERCENTILE = 99.5
INPAINT_RADIUS = 10
JPEG_QUALITY = 95
PREDICTION_THRESHOLD = 0.5

# mixed0 表示较浅的编码特征，mixed10 表示 InceptionV3 深层瓶颈特征。
FEATURE_LAYERS = ("mixed0", "mixed10")
SOURCE_BATCH_SIZE = 8
INFERENCE_BATCH_SIZE = 4
NOISE_REPEATS = 3
NOISE_ENERGY_TOLERANCE = 0.005
FEATURE_MASK_DILATION = 3
CASE_TOP_K = 3
FIGURE_DPI = 300
SEED = 42
OVERWRITE_CACHE = False

NOTEBOOK_DIR = Path.cwd().resolve()
RESULT_DIR = (
    NOTEBOOK_DIR / "simple_feature_interference_results" / SAFE_MODEL_NAME
)
CACHE_DIR = RESULT_DIR / "cache"
FIGURE_DIR = RESULT_DIR / "figures"
CASE_FIGURE_DIR = FIGURE_DIR / "representative_cases"
for directory in (RESULT_DIR, CACHE_DIR, FIGURE_DIR, CASE_FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

TEST_ROOT = PROJECT_ROOT / "Data" / "Test"
WEIGHT_ROOT = PROJECT_ROOT / "best weight" / "2.different mode"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

ANALYSIS_SIGNATURE = {
    "models": list(MODEL_CONFIGS),
    "datasets": list(TEST_DATASETS),
    "image_size": list(IMAGE_SIZE),
    "percentile": PERCENTILE,
    "inpaint_method": "cv2.INPAINT_TELEA",
    "inpaint_radius": INPAINT_RADIUS,
    "fmm_scale": 0.6,
    "jpeg_quality": JPEG_QUALITY,
    "feature_layers": list(FEATURE_LAYERS),
    "noise_repeats": NOISE_REPEATS,
    "seed": SEED,
}

signature_path = RESULT_DIR / "analysis_signature.json"
if signature_path.exists() and not OVERWRITE_CACHE:
    existing_signature = json.loads(signature_path.read_text(encoding="utf-8"))
    if existing_signature != ANALYSIS_SIGNATURE:
        raise RuntimeError(
            "已有缓存的分析参数与当前配置不同。请设置 OVERWRITE_CACHE=True，"
            "或删除 simple_feature_interference_results 后重新运行。"
        )
signature_path.write_text(
    json.dumps(ANALYSIS_SIGNATURE, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Project root:", PROJECT_ROOT.resolve())
print("Result directory:", RESULT_DIR.resolve())
print("Model:", MODEL_NAME)


Segmentation Models: using `tf.keras` framework.
Project root: D:\WJY\paper1
Result directory: C:\Users\w\Raw-vs-FMM-Train-Inference-Wilcoxon-PairedT\bad-case\simple_feature_interference_results\FPN+V3
Model: FPN+V3


In [2]:
# ============================================================
# 2. 模型建立与 Raw 权重的严格解析
# ============================================================
def build_segmentation_model(model_name):
    """按原实验建立 FPN+V3，不下载新权重。"""
    if model_name != MODEL_NAME:
        raise ValueError(f"Unexpected model: {model_name}")
    return sm.FPN(
        "inceptionv3",
        classes=1,
        activation="sigmoid",
        encoder_weights=None,
    )


def resolve_raw_weight(model_name):
    """优先读取统一命名；历史目录只能存在一个 H5 候选。"""
    raw_dir = WEIGHT_ROOT / model_name / "img"
    standard = raw_dir / f"{model_name}_img_bestweight.h5"
    if standard.is_file():
        return standard
    candidates = sorted(path for path in raw_dir.glob("*.h5") if path.is_file())
    if len(candidates) == 1:
        print(f"{model_name}: using historical Raw weight {candidates[0].resolve()}")
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(
            f"No Raw weight found for {model_name}. Standard path: {standard}"
        )
    candidate_text = "\n".join(str(path.resolve()) for path in candidates)
    raise RuntimeError(
        f"Multiple Raw weights found for {model_name}; selection is ambiguous:\n"
        f"{candidate_text}"
    )


def build_feature_model(model_name):
    """同时输出浅层、深层特征和最终分割概率。"""
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    segmentation_model = build_segmentation_model(model_name)
    weight_path = resolve_raw_weight(model_name)
    try:
        segmentation_model.load_weights(str(weight_path))
    except Exception as exc:
        raise RuntimeError(
            f"Cannot fully load {model_name} Raw weights: {weight_path}"
        ) from exc

    feature_outputs = []
    for layer_name in FEATURE_LAYERS:
        try:
            output = segmentation_model.get_layer(layer_name).output
        except ValueError as exc:
            available = [layer.name for layer in segmentation_model.layers]
            raise RuntimeError(
                f"{model_name} does not contain feature layer {layer_name}. "
                f"Available layer examples: {available[:30]}"
            ) from exc
        if len(output.shape) != 4:
            raise ValueError(
                f"Feature layer {layer_name} is not spatial: {output.shape}"
            )
        feature_outputs.append(output)

    feature_model = tf.keras.Model(
        segmentation_model.input,
        feature_outputs + [segmentation_model.output],
        name=f"{model_name.replace('+', '_')}_feature_extractor",
    )

    # 在正式分析前用固定尺寸零张量检查模型图，及早发现架构、层名或输出尺寸错误。
    dummy_input = tf.zeros((1, IMAGE_HEIGHT, IMAGE_WIDTH, 3), dtype=tf.float32)
    dummy_outputs = feature_model(dummy_input, training=False)
    if len(dummy_outputs) != len(FEATURE_LAYERS) + 1:
        raise AssertionError(
            f"Unexpected output count: {len(dummy_outputs)}"
        )
    for layer_name, feature in zip(FEATURE_LAYERS, dummy_outputs[:-1]):
        if len(feature.shape) != 4 or int(feature.shape[0]) != 1:
            raise AssertionError(
                f"Invalid feature shape for {layer_name}: {feature.shape}"
            )
        if not bool(tf.reduce_all(tf.math.is_finite(feature)).numpy()):
            raise FloatingPointError(
                f"Non-finite dummy feature from {layer_name}"
            )
    segmentation_shape = tuple(int(value) for value in dummy_outputs[-1].shape)
    expected_shape = (1, IMAGE_HEIGHT, IMAGE_WIDTH, 1)
    if segmentation_shape != expected_shape:
        raise AssertionError(
            f"Unexpected segmentation output: {segmentation_shape} != {expected_shape}"
        )
    early_shape = dummy_outputs[0].shape
    deep_shape = dummy_outputs[1].shape
    if not (
        int(early_shape[1]) > int(deep_shape[1])
        and int(early_shape[2]) > int(deep_shape[2])
    ):
        raise AssertionError(
            f"Feature hierarchy is unexpected: {early_shape}, {deep_shape}"
        )
    del dummy_input, dummy_outputs

    print(model_name, "Raw weight:", weight_path.resolve())
    print(
        model_name,
        "outputs:",
        [tuple(output.shape) for output in feature_model.outputs],
    )
    return feature_model


In [3]:
# ============================================================
# 3. 数据读取与严格复现的 99.5% + FMM(r=10)
# ============================================================
def list_image_names(image_dir):
    """保持原实验的 os.listdir 顺序，并检查目录非空。"""
    names = [
        name
        for name in os.listdir(image_dir)
        if (Path(image_dir) / name).is_file()
        and Path(name).suffix.lower() in IMAGE_EXTENSIONS
    ]
    if not names:
        raise FileNotFoundError(f"No images found in {image_dir}")
    return tuple(names)


def read_raw_image(image_path):
    """复现Raw分支：直接缩放到384x288、BGR转RGB、归一化到[0, 1]。"""
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f"Cannot read image: {image_path}")
    image = cv2.resize(image, IMAGE_SIZE)
    image = image[..., ::-1]  # BGR to RGB
    return image.astype(np.float32) / 255.0


def create_highlight_mask(image_path, percentile=PERCENTILE):
    """按灰度图的99.5%分位值生成二值高光mask。"""
    gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise ValueError(f"Cannot read grayscale image: {image_path}")
    # 与原notebook一致：阈值T=P_99.5(I_g)，THRESH_BINARY保留严格大于T的像素。
    threshold = np.percentile(gray, percentile)
    _, mask = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
    return mask


def read_fmm_image(image_path):
    """严格复现：99.5%高光mask + TELEA/FMM修复半径10 + JPEG质量95。"""
    source = cv2.imread(str(image_path))
    if source is None:
        raise ValueError(f"Cannot read image: {image_path}")
    # 步骤1：在原始灰度图上计算99.5%分位阈值并生成高光mask。
    highlight_mask = create_highlight_mask(image_path, percentile=PERCENTILE)

    # 步骤2：原图和高光mask均使用INTER_CUBIC缩小到0.6倍。
    # 重要：mask缩放后不再次二值化，这一点严格保留原notebook行为。
    resized_source = cv2.resize(
        source, None, fx=0.6, fy=0.6, interpolation=cv2.INTER_CUBIC
    )
    resized_mask = cv2.resize(
        highlight_mask, None, fx=0.6, fy=0.6, interpolation=cv2.INTER_CUBIC
    )
    # 步骤3：使用OpenCV TELEA实现的Fast Marching Method，修复半径r=10。
    restored = cv2.inpaint(
        resized_source, resized_mask, INPAINT_RADIUS, cv2.INPAINT_TELEA
    )
    # 步骤4：按原代码默认的线性插值恢复到384x288。
    restored = cv2.resize(restored, IMAGE_SIZE)

    # 步骤5：原流程会将修复图保存为JPG后再读取；此处以内存编码复现，
    # 质量固定为95，不生成中间文件。
    encoded_ok, encoded = cv2.imencode(
        ".jpg", restored, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY]
    )
    if not encoded_ok:
        raise RuntimeError(f"Cannot encode FMM image: {image_path}")
    restored = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
    # 步骤6：JPG重新读取后为BGR，转换为RGB并归一化到[0, 1]。
    restored = restored[..., ::-1]
    return restored.astype(np.float32) / 255.0


def read_resized_highlight_mask(image_path):
    """分析 mask 使用最近邻缩放，避免改变二值区域。"""
    mask = create_highlight_mask(image_path)
    return cv2.resize(mask, IMAGE_SIZE, interpolation=cv2.INTER_NEAREST) > 0


def read_ground_truth(mask_path):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise ValueError(f"Cannot read mask: {mask_path}")
    mask = cv2.resize(mask, IMAGE_SIZE)
    return (mask.astype(np.float32) / 255.0).reshape(
        IMAGE_HEIGHT, IMAGE_WIDTH, 1
    )


def validate_configuration():
    observed = {
        "percentile": PERCENTILE,
        "radius": INPAINT_RADIUS,
        "jpeg": JPEG_QUALITY,
        "size": IMAGE_SIZE,
    }
    expected = {
        "percentile": 99.5,
        "radius": 10,
        "jpeg": 95,
        "size": (384, 288),
    }
    if observed != expected:
        raise AssertionError(f"FMM configuration changed: {observed} != {expected}")
    for dataset_name in TEST_DATASETS:
        image_dir = TEST_ROOT / dataset_name / "img"
        mask_dir = TEST_ROOT / dataset_name / "masks"
        names = list_image_names(image_dir)
        missing = [name for name in names if not (mask_dir / name).is_file()]
        if missing:
            raise FileNotFoundError(
                f"{dataset_name} is missing {len(missing)} GT masks: {missing[:5]}"
            )
    print(
        "Preflight passed: P99.5 mask, scale=0.6, TELEA, r=10, "
        "JPEG=95, size=384x288."
    )


validate_configuration()


Preflight passed: P99.5 mask, scale=0.6, TELEA, r=10, JPEG=95, size=384x288.


## 4. 构造公平的结构化高光与匹配噪声对照

为避免 Raw 与 FMM 两条完整预处理链的全局差异混入比较，本分析不直接把整张 Raw
与整张 FMM 相减。我们从 FMM 图像出发，只在二值高光 mask 内重新放回 Raw 像素，
得到 `Highlight` 条件。随机噪声也只作用于同一个 mask，并通过二分缩放使裁剪后的
实际 L2 能量与真实高光残差匹配。这样比较的变量主要是扰动的结构，而不是面积或能量。


In [4]:
# ============================================================
# 4. 配对输入与能量匹配噪声
# ============================================================
def deterministic_rng(dataset_name, image_name, repeat_index):
    token = f"{SEED}|{dataset_name}|{image_name}|{repeat_index}".encode("utf-8")
    return np.random.default_rng(zlib.crc32(token) & 0xFFFFFFFF)


def make_highlight_condition(raw_image, fmm_image, highlight_mask):
    """只在高光区域重新嵌入 Raw 像素，其他位置完全保持 FMM。"""
    structured = fmm_image.copy()
    structured[highlight_mask] = raw_image[highlight_mask]
    return structured


def make_energy_matched_noise(fmm_image, residual, highlight_mask, rng):
    """在相同 mask 内生成实现后 L2 能量匹配的随机噪声。"""
    base_values = fmm_image[highlight_mask].astype(np.float64)
    residual_values = residual[highlight_mask].astype(np.float64)
    target_energy = float(np.linalg.norm(residual_values))
    if len(base_values) == 0 or target_energy <= 1e-12:
        return fmm_image.copy(), np.nan

    channel_mean = residual_values.mean(axis=0)
    channel_std = residual_values.std(axis=0)
    channel_std = np.maximum(channel_std, 1e-6)

    gaussian = rng.normal(size=residual_values.shape)
    gaussian = (
        gaussian - gaussian.mean(axis=0, keepdims=True)
    ) / np.maximum(gaussian.std(axis=0, keepdims=True), 1e-12)
    candidate = channel_mean + channel_std * gaussian

    def realized(scale):
        changed = np.clip(base_values + scale * candidate, 0.0, 1.0)
        delta = changed - base_values
        return changed, float(np.linalg.norm(delta))

    low, high = 0.0, 1.0
    changed, energy = realized(high)
    while energy < target_energy and high < 1024.0:
        high *= 2.0
        changed, energy = realized(high)
    if energy < target_energy * (1.0 - NOISE_ENERGY_TOLERANCE):
        raise RuntimeError(
            "Cannot create an energy-matched noise control after clipping."
        )

    for _ in range(50):
        middle = (low + high) / 2.0
        changed, energy = realized(middle)
        if energy < target_energy:
            low = middle
        else:
            high = middle
    changed, energy = realized(high)

    noise_image = fmm_image.copy()
    noise_image[highlight_mask] = changed.astype(np.float32)
    energy_ratio = energy / target_energy
    if abs(energy_ratio - 1.0) > NOISE_ENERGY_TOLERANCE:
        raise AssertionError(
            f"Noise energy mismatch: ratio={energy_ratio:.6f}"
        )
    return noise_image, float(energy_ratio)


def build_input_variants(dataset_name, image_name):
    image_path = TEST_ROOT / dataset_name / "img" / image_name
    mask_path = TEST_ROOT / dataset_name / "masks" / image_name
    raw = read_raw_image(image_path)
    fmm = read_fmm_image(image_path)
    highlight_mask = read_resized_highlight_mask(image_path)
    ground_truth = read_ground_truth(mask_path)
    structured = make_highlight_condition(raw, fmm, highlight_mask)
    residual = structured - fmm

    noises, energy_ratios = [], []
    for repeat_index in range(NOISE_REPEATS):
        noise, ratio = make_energy_matched_noise(
            fmm,
            residual,
            highlight_mask,
            deterministic_rng(dataset_name, image_name, repeat_index),
        )
        noises.append(noise)
        energy_ratios.append(ratio)

    return {
        "raw": raw,
        "fmm": fmm,
        "highlight": structured,
        "noises": noises,
        "highlight_mask": highlight_mask,
        "ground_truth": ground_truth,
        "noise_energy_ratios": energy_ratios,
    }


In [5]:
# ============================================================
# 5. 特征、空间传播和预测一致性指标
# ============================================================
def relative_l2(reference, changed):
    reference64 = reference.astype(np.float64)
    changed64 = changed.astype(np.float64)
    return float(
        np.linalg.norm(changed64 - reference64)
        / max(np.linalg.norm(reference64), 1e-12)
    )


def cosine_similarity(reference, changed):
    a = reference.astype(np.float64).ravel()
    b = changed.astype(np.float64).ravel()
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / max(denominator, 1e-12))


def feature_difference_map(reference, changed):
    return np.mean(
        np.abs(changed.astype(np.float32) - reference.astype(np.float32)),
        axis=-1,
    )


def outside_mask_shift_ratio(reference, changed, highlight_mask):
    """特征变化中落在膨胀高光区域之外的比例，越高表示传播越非局部。"""
    difference = feature_difference_map(reference, changed)
    total = float(difference.sum())
    if total <= 1e-12:
        return 0.0
    kernel_size = 2 * FEATURE_MASK_DILATION + 1
    dilated = cv2.dilate(
        highlight_mask.astype(np.uint8),
        np.ones((kernel_size, kernel_size), np.uint8),
        iterations=1,
    )
    feature_mask = cv2.resize(
        dilated.astype(np.float32),
        (difference.shape[1], difference.shape[0]),
        interpolation=cv2.INTER_AREA,
    ) > 0.0
    return float(difference[~feature_mask].sum() / total)


def binary_prediction_dice(probability_a, probability_b):
    a = probability_a[..., 0] >= PREDICTION_THRESHOLD
    b = probability_b[..., 0] >= PREDICTION_THRESHOLD
    intersection = int(np.logical_and(a, b).sum())
    denominator = int(a.sum() + b.sum())
    return float(2.0 * intersection / denominator) if denominator else 1.0


def ground_truth_to_2d(ground_truth):
    """统一接受(H,W)或(H,W,1) GT，返回严格的二维数组。"""
    ground_truth = np.asarray(ground_truth)
    if ground_truth.ndim == 3 and ground_truth.shape[-1] == 1:
        return ground_truth[..., 0]
    if ground_truth.ndim == 2:
        return ground_truth
    raise ValueError(
        f"Ground-truth shape must be (H,W) or (H,W,1), got {ground_truth.shape}"
    )


def soft_dice_to_ground_truth(ground_truth, probability, smooth=1e-7):
    gt = ground_truth_to_2d(ground_truth).astype(np.float64)
    probability = np.asarray(probability)
    if probability.ndim != 3 or probability.shape[-1] != 1:
        raise ValueError(
            f"Probability shape must be (H,W,1), got {probability.shape}"
        )
    pred = probability[..., 0].astype(np.float64)
    if gt.shape != pred.shape:
        raise ValueError(f"GT/prediction shape mismatch: {gt.shape} != {pred.shape}")
    intersection = float(np.sum(gt * pred))
    return float(
        (2.0 * intersection + smooth)
        / (float(gt.sum()) + float(pred.sum()) + smooth)
    )


def prediction_metrics(fmm_probability, changed_probability, ground_truth):
    return {
        "probability_mae": float(
            np.mean(np.abs(changed_probability - fmm_probability))
        ),
        "prediction_dice_to_fmm": binary_prediction_dice(
            fmm_probability, changed_probability
        ),
        "soft_dice_gt": soft_dice_to_ground_truth(
            ground_truth, changed_probability
        ),
    }


def paired_rank_biserial(differences):
    differences = np.asarray(differences, dtype=np.float64)
    differences = differences[np.isfinite(differences) & (differences != 0)]
    if not len(differences):
        return 0.0
    ranks = rankdata(np.abs(differences), method="average")
    positive = float(ranks[differences > 0].sum())
    negative = float(ranks[differences < 0].sum())
    return float((positive - negative) / max(positive + negative, 1e-12))


def safe_spearman(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 3 or np.unique(x[valid]).size < 2 or np.unique(y[valid]).size < 2:
        return np.nan, np.nan
    result = spearmanr(x[valid], y[valid])
    return float(result.statistic), float(result.pvalue)


In [6]:
# ============================================================
# 6. 按数据集提取特征并增量保存
# ============================================================
def analyze_dataset(model_name, feature_model, dataset_name):
    cache_model_dir = CACHE_DIR / re.sub(r"[^A-Za-z0-9._+-]+", "_", model_name)
    cache_model_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_model_dir / f"{dataset_name}.csv"
    if cache_path.exists() and not OVERWRITE_CACHE:
        print(model_name, dataset_name, ": loading cache")
        return pd.read_csv(cache_path)

    image_dir = TEST_ROOT / dataset_name / "img"
    names = list_image_names(image_dir)
    rows = []
    print(model_name, dataset_name, ":", len(names), "images")

    for start in range(0, len(names), SOURCE_BATCH_SIZE):
        selected_names = names[start:start + SOURCE_BATCH_SIZE]
        model_inputs = []
        sample_records = []

        for image_name in selected_names:
            variants = build_input_variants(dataset_name, image_name)
            if not variants["highlight_mask"].any():
                print("  skip empty highlight mask:", dataset_name, image_name)
                continue

            indices = {"fmm": len(model_inputs)}
            model_inputs.append(variants["fmm"])
            indices["highlight"] = len(model_inputs)
            model_inputs.append(variants["highlight"])
            indices["noises"] = []
            for noise in variants["noises"]:
                indices["noises"].append(len(model_inputs))
                model_inputs.append(noise)
            sample_records.append((image_name, variants, indices))

        if not model_inputs:
            continue
        outputs = feature_model.predict(
            np.asarray(model_inputs, dtype=np.float32),
            batch_size=INFERENCE_BATCH_SIZE,
            verbose=0,
        )
        if len(outputs) != len(FEATURE_LAYERS) + 1:
            raise RuntimeError("Unexpected feature model output count")
        if not all(np.isfinite(output).all() for output in outputs):
            raise FloatingPointError("Feature model output contains NaN or Inf")

        feature_outputs = {
            layer_name: outputs[index]
            for index, layer_name in enumerate(FEATURE_LAYERS)
        }
        probabilities = outputs[-1]

        for image_name, variants, indices in sample_records:
            fmm_index = indices["fmm"]
            highlight_index = indices["highlight"]
            noise_indices = indices["noises"]
            fmm_probability = probabilities[fmm_index]
            highlight_probability = probabilities[highlight_index]
            noise_probabilities = probabilities[noise_indices]

            highlight_prediction = prediction_metrics(
                fmm_probability,
                highlight_probability,
                variants["ground_truth"],
            )
            noise_prediction_values = [
                prediction_metrics(
                    fmm_probability,
                    noise_probability,
                    variants["ground_truth"],
                )
                for noise_probability in noise_probabilities
            ]
            noise_prediction = {
                key: float(np.mean([item[key] for item in noise_prediction_values]))
                for key in noise_prediction_values[0]
            }
            fmm_soft_dice = soft_dice_to_ground_truth(
                variants["ground_truth"], fmm_probability
            )

            input_highlight_l2 = relative_l2(
                variants["fmm"], variants["highlight"]
            )
            input_noise_l2 = float(np.mean([
                relative_l2(variants["fmm"], noise)
                for noise in variants["noises"]
            ]))

            for layer_name in FEATURE_LAYERS:
                layer_output = feature_outputs[layer_name]
                fmm_feature = layer_output[fmm_index]
                highlight_feature = layer_output[highlight_index]
                noise_features = layer_output[noise_indices]

                highlight_feature_l2 = relative_l2(
                    fmm_feature, highlight_feature
                )
                noise_feature_l2_values = [
                    relative_l2(fmm_feature, noise_feature)
                    for noise_feature in noise_features
                ]
                highlight_cosine = cosine_similarity(
                    fmm_feature, highlight_feature
                )
                noise_cosine_values = [
                    cosine_similarity(fmm_feature, noise_feature)
                    for noise_feature in noise_features
                ]
                highlight_outside = outside_mask_shift_ratio(
                    fmm_feature,
                    highlight_feature,
                    variants["highlight_mask"],
                )
                noise_outside_values = [
                    outside_mask_shift_ratio(
                        fmm_feature,
                        noise_feature,
                        variants["highlight_mask"],
                    )
                    for noise_feature in noise_features
                ]

                rows.append({
                    "model": model_name,
                    "dataset": dataset_name,
                    "image_name": image_name,
                    "layer": layer_name,
                    "highlight_area_ratio": float(
                        variants["highlight_mask"].mean()
                    ),
                    "input_highlight_relative_l2": input_highlight_l2,
                    "input_noise_relative_l2": input_noise_l2,
                    "noise_energy_ratio": float(np.nanmean(
                        variants["noise_energy_ratios"]
                    )),
                    "highlight_feature_relative_l2": highlight_feature_l2,
                    "noise_feature_relative_l2": float(np.mean(
                        noise_feature_l2_values
                    )),
                    "delta_feature_l2_highlight_minus_noise": float(
                        highlight_feature_l2 - np.mean(noise_feature_l2_values)
                    ),
                    "highlight_feature_cosine": highlight_cosine,
                    "noise_feature_cosine": float(np.mean(noise_cosine_values)),
                    "highlight_outside_mask_shift_ratio": highlight_outside,
                    "noise_outside_mask_shift_ratio": float(np.mean(
                        noise_outside_values
                    )),
                    "fmm_soft_dice_gt": fmm_soft_dice,
                    "highlight_soft_dice_gt": highlight_prediction["soft_dice_gt"],
                    "noise_soft_dice_gt": noise_prediction["soft_dice_gt"],
                    "delta_highlight_vs_fmm_soft_dice": float(
                        highlight_prediction["soft_dice_gt"] - fmm_soft_dice
                    ),
                    "highlight_probability_mae": highlight_prediction[
                        "probability_mae"
                    ],
                    "noise_probability_mae": noise_prediction["probability_mae"],
                    "highlight_prediction_dice_to_fmm": highlight_prediction[
                        "prediction_dice_to_fmm"
                    ],
                    "noise_prediction_dice_to_fmm": noise_prediction[
                        "prediction_dice_to_fmm"
                    ],
                })

        print(
            f"  processed {min(start + SOURCE_BATCH_SIZE, len(names))}/{len(names)}"
        )
        del outputs, feature_outputs, probabilities, model_inputs
        gc.collect()

    table = pd.DataFrame(rows)
    if table.empty:
        raise RuntimeError(f"No analyzable images in {dataset_name}")
    table.to_csv(cache_path, index=False, encoding="utf-8-sig")
    return table


all_tables = []
for current_model_name in MODEL_CONFIGS:
    print("\nAnalyzing", current_model_name)
    current_feature_model = build_feature_model(current_model_name)
    for current_dataset_name in TEST_DATASETS:
        all_tables.append(analyze_dataset(
            current_model_name,
            current_feature_model,
            current_dataset_name,
        ))
    del current_feature_model
    tf.keras.backend.clear_session()
    gc.collect()

PER_IMAGE_TABLE = pd.concat(all_tables, ignore_index=True)
PER_IMAGE_CSV = RESULT_DIR / "per_image_feature_interference.csv"
PER_IMAGE_TABLE.to_csv(PER_IMAGE_CSV, index=False, encoding="utf-8-sig")
print("Per-image table:", PER_IMAGE_CSV.resolve())
display(PER_IMAGE_TABLE.head().round(5))



Analyzing FPN+V3
FPN+V3 Raw weight: D:\WJY\paper1\best weight\2.different mode\FPN+V3\img\FPN+V3_img_bestweight.h5
FPN+V3 outputs: [(None, None, None, 256), (None, None, None, 2048), (None, None, None, 1)]
FPN+V3 CVC-300 : 60 images
  processed 8/60
  processed 16/60
  processed 24/60
  processed 32/60
  processed 40/60
  processed 48/60
  processed 56/60
  processed 60/60
FPN+V3 CVC-ClinicDB : 62 images
  processed 8/62
  processed 16/62
  processed 24/62
  processed 32/62
  processed 40/62
  processed 48/62
  processed 56/62
  processed 62/62
FPN+V3 CVC-ColonDB : 380 images
  processed 8/380
  processed 16/380
  processed 24/380
  processed 32/380
  processed 40/380
  processed 48/380
  processed 56/380
  processed 64/380
  processed 72/380
  processed 80/380
  processed 88/380
  processed 96/380
  processed 104/380
  processed 112/380
  processed 120/380
  processed 128/380
  processed 136/380
  processed 144/380
  processed 152/380
  processed 160/380
  processed 168/380
  process

,model,dataset,image_name,layer,highlight_area_ratio,input_highlight_relative_l2,input_noise_relative_l2,noise_energy_ratio,highlight_feature_relative_l2,noise_feature_relative_l2,...,highlight_outside_mask_shift_ratio,noise_outside_mask_shift_ratio,fmm_soft_dice_gt,highlight_soft_dice_gt,noise_soft_dice_gt,delta_highlight_vs_fmm_soft_dice,highlight_probability_mae,noise_probability_mae,highlight_prediction_dice_to_fmm,noise_prediction_dice_to_fmm
0,FPN+V3,CVC-300,149.png,mixed0,0.00505,0.04253,0.04253,1.0,0.08591,0.08613,...,0.46793,0.46404,0.95076,0.94488,0.94584,-0.00587,0.00188,0.00166,0.98985,0.99060
1,FPN+V3,CVC-300,149.png,mixed10,0.00505,0.04253,0.04253,1.0,0.08609,0.07986,...,0.71961,0.72315,0.95076,0.94488,0.94584,-0.00587,0.00188,0.00166,0.98985,0.99060
2,FPN+V3,CVC-300,150.png,mixed0,0.00487,0.05793,0.05793,1.0,0.11140,0.10768,...,0.39795,0.40988,0.87302,0.80691,0.80794,-0.06611,0.01419,0.01390,0.92029,0.92066
3,FPN+V3,CVC-300,150.png,mixed10,0.00487,0.05793,0.05793,1.0,0.09084,0.08537,...,0.63997,0.62360,0.87302,0.80691,0.80794,-0.06611,0.01419,0.01390,0.92029,0.92066
4,FPN+V3,CVC-300,151.png,mixed0,0.00505,0.05098,0.05098,1.0,0.09536,0.09452,...,0.43171,0.43324,0.90105,0.90127,0.90049,0.00022,0.00035,0.00041,0.99639,0.99539


In [7]:
# ============================================================
# 7. 配对统计、效应量和数据集汇总
# ============================================================
def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=np.float64)
    adjusted = np.full_like(p_values, np.nan)
    valid_indices = np.where(np.isfinite(p_values))[0]
    if not len(valid_indices):
        return adjusted
    valid = p_values[valid_indices]
    order = np.argsort(valid)
    ordered = valid[order]
    ordered_adjusted = np.maximum.accumulate(
        (len(ordered) - np.arange(len(ordered))) * ordered
    )
    restored = np.empty_like(ordered_adjusted)
    restored[order] = np.minimum(ordered_adjusted, 1.0)
    adjusted[valid_indices] = restored
    return adjusted


def summarize_group(model_name, dataset_name, layer_name, group):
    highlight = group["highlight_feature_relative_l2"].to_numpy()
    noise = group["noise_feature_relative_l2"].to_numpy()
    differences = highlight - noise
    if np.allclose(differences, 0.0):
        wilcoxon_p = 1.0
    else:
        wilcoxon_p = float(
            wilcoxon(highlight, noise, zero_method="wilcox").pvalue
        )
    rho, rho_p = safe_spearman(
        highlight,
        group["highlight_probability_mae"].to_numpy(),
    )
    return {
        "model": model_name,
        "dataset": dataset_name,
        "layer": layer_name,
        "N": len(group),
        "highlight_feature_l2_median": float(np.median(highlight)),
        "highlight_feature_l2_q1": float(np.quantile(highlight, 0.25)),
        "highlight_feature_l2_q3": float(np.quantile(highlight, 0.75)),
        "noise_feature_l2_median": float(np.median(noise)),
        "noise_feature_l2_q1": float(np.quantile(noise, 0.25)),
        "noise_feature_l2_q3": float(np.quantile(noise, 0.75)),
        "median_delta_highlight_minus_noise": float(np.median(differences)),
        "highlight_greater_than_noise_rate": float(np.mean(differences > 0)),
        "paired_rank_biserial": paired_rank_biserial(differences),
        "wilcoxon_p": wilcoxon_p,
        "highlight_cosine_median": float(np.median(
            group["highlight_feature_cosine"]
        )),
        "noise_cosine_median": float(np.median(
            group["noise_feature_cosine"]
        )),
        "highlight_outside_ratio_median": float(np.median(
            group["highlight_outside_mask_shift_ratio"]
        )),
        "noise_outside_ratio_median": float(np.median(
            group["noise_outside_mask_shift_ratio"]
        )),
        "highlight_probability_mae_median": float(np.median(
            group["highlight_probability_mae"]
        )),
        "noise_probability_mae_median": float(np.median(
            group["noise_probability_mae"]
        )),
        "highlight_prediction_dice_median": float(np.median(
            group["highlight_prediction_dice_to_fmm"]
        )),
        "noise_prediction_dice_median": float(np.median(
            group["noise_prediction_dice_to_fmm"]
        )),
        "rho_feature_shift_vs_prediction_mae": rho,
        "rho_p_value": rho_p,
    }


summary_rows = []
for model_name in MODEL_CONFIGS:
    model_table = PER_IMAGE_TABLE[PER_IMAGE_TABLE["model"] == model_name]
    for layer_name in FEATURE_LAYERS:
        layer_table = model_table[model_table["layer"] == layer_name]
        for dataset_name in TEST_DATASETS:
            group = layer_table[layer_table["dataset"] == dataset_name]
            summary_rows.append(summarize_group(
                model_name, dataset_name, layer_name, group
            ))
        summary_rows.append(summarize_group(
            model_name, "ALL", layer_name, layer_table
        ))

SUMMARY_TABLE = pd.DataFrame(summary_rows)
SUMMARY_TABLE["wilcoxon_p_holm"] = np.nan
for (model_name, layer_name), indices in SUMMARY_TABLE.groupby(
    ["model", "layer"]
).groups.items():
    indices = list(indices)
    dataset_indices = [
        index for index in indices
        if SUMMARY_TABLE.loc[index, "dataset"] != "ALL"
    ]
    SUMMARY_TABLE.loc[dataset_indices, "wilcoxon_p_holm"] = holm_adjust(
        SUMMARY_TABLE.loc[dataset_indices, "wilcoxon_p"].to_numpy()
    )

STATISTICS_CSV = RESULT_DIR / "feature_interference_statistics.csv"
SUMMARY_XLSX = RESULT_DIR / "feature_interference_summary.xlsx"
SUMMARY_TABLE.to_csv(STATISTICS_CSV, index=False, encoding="utf-8-sig")
with pd.ExcelWriter(SUMMARY_XLSX, engine="openpyxl") as writer:
    SUMMARY_TABLE.to_excel(writer, sheet_name="Summary", index=False)
    PER_IMAGE_TABLE.to_excel(writer, sheet_name="Per Image Long", index=False)

display(SUMMARY_TABLE.round(5))
print("Statistics:", STATISTICS_CSV.resolve())
print("Workbook:", SUMMARY_XLSX.resolve())


,model,dataset,layer,N,highlight_feature_l2_median,highlight_feature_l2_q1,highlight_feature_l2_q3,noise_feature_l2_median,noise_feature_l2_q1,noise_feature_l2_q3,...,noise_cosine_median,highlight_outside_ratio_median,noise_outside_ratio_median,highlight_probability_mae_median,noise_probability_mae_median,highlight_prediction_dice_median,noise_prediction_dice_median,rho_feature_shift_vs_prediction_mae,rho_p_value,wilcoxon_p_holm
0,FPN+V3,CVC-300,mixed0,60,0.06127,0.04726,0.06846,0.05594,0.04587,0.06056,...,0.99843,0.41924,0.41661,0.00047,0.00044,0.98988,0.99217,0.37679,0.00300,0.00059
1,FPN+V3,CVC-ClinicDB,mixed0,62,0.08753,0.05860,0.10042,0.08332,0.06019,0.09914,...,0.99654,0.45547,0.45910,0.00120,0.00118,0.99172,0.99153,0.32890,0.00905,0.00017
2,FPN+V3,CVC-ColonDB,mixed0,379,0.05994,0.04434,0.07849,0.05709,0.04282,0.07259,...,0.99837,0.44716,0.44624,0.00052,0.00053,0.99085,0.99097,0.29234,0.00000,0.00000
3,FPN+V3,ETIS-LaribPolypDB,mixed0,195,0.02675,0.00319,0.05731,0.02751,0.00254,0.05655,...,0.99962,0.47029,0.46045,0.00012,0.00014,0.99697,0.99655,0.61249,0.00000,0.00000
4,FPN+V3,Kvasir,mixed0,100,0.04220,0.01641,0.06062,0.03791,0.01592,0.05893,...,0.99928,0.43414,0.42279,0.00067,0.00063,0.99644,0.99680,0.49462,0.00000,0.00000
5,FPN+V3,ALL,mixed0,796,0.05466,0.02960,0.07471,0.05238,0.02862,0.07028,...,0.99863,0.44770,0.44501,0.00051,0.00052,0.99346,0.99346,0.42282,0.00000,NaN
6,FPN+V3,CVC-300,mixed10,60,0.02968,0.01376,0.05924,0.02879,0.01419,0.05961,...,0.99958,0.78292,0.77593,0.00047,0.00044,0.98988,0.99217,0.77926,0.00000,0.21076
7,FPN+V3,CVC-ClinicDB,mixed10,62,0.08285,0.05356,0.10651,0.08174,0.05047,0.10641,...,0.99692,0.72009,0.71966,0.00120,0.00118,0.99172,0.99153,0.68699,0.00000,0.00333
8,FPN+V3,CVC-ColonDB,mixed10,379,0.04327,0.02035,0.07827,0.03908,0.01949,0.06869,...,0.99927,0.76485,0.75938,0.00052,0.00053,0.99085,0.99097,0.61504,0.00000,0.00000
9,FPN+V3,ETIS-LaribPolypDB,mixed10,195,0.02259,0.00262,0.07517,0.01889,0.00244,0.06752,...,0.99982,0.73456,0.72225,0.00012,0.00014,0.99697,0.99655,0.68415,0.00000,0.00000


Statistics: C:\Users\w\Raw-vs-FMM-Train-Inference-Wilcoxon-PairedT\bad-case\simple_feature_interference_results\FPN+V3\feature_interference_statistics.csv
Workbook: C:\Users\w\Raw-vs-FMM-Train-Inference-Wilcoxon-PairedT\bad-case\simple_feature_interference_results\FPN+V3\feature_interference_summary.xlsx


In [8]:
# ============================================================
# 8. 总体可视化：特征偏移、层级变化、预测一致性和非局部传播
# ============================================================
MODEL_COLORS = {MODEL_NAME: "#2A6F97"}
CONDITION_COLORS = {"Highlight": "#D55E00", "Matched noise": "#309E74"}
DATASET_COLORS = {
    "CVC-300": "#0072B2",
    "CVC-ClinicDB": "#E69F00",
    "CVC-ColonDB": "#009E73",
    "ETIS-LaribPolypDB": "#CC79A7",
    "Kvasir": "#6B6B6B",
}


def save_feature_shift_boxplot():
    figure, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), sharey=True)
    for axis, layer_name in zip(axes, FEATURE_LAYERS):
        group = PER_IMAGE_TABLE[PER_IMAGE_TABLE["layer"] == layer_name]
        values = [
            group["highlight_feature_relative_l2"],
            group["noise_feature_relative_l2"],
        ]
        boxes = axis.boxplot(
            values,
            labels=["Highlight", "Matched\nnoise"],
            patch_artist=True,
            showfliers=False,
        )
        for box, condition in zip(boxes["boxes"], CONDITION_COLORS):
            box.set_facecolor(CONDITION_COLORS[condition])
            box.set_alpha(0.72)
        axis.set_title(f"{MODEL_NAME} | {layer_name}")
        axis.set_ylabel("Relative feature L2 shift")
        axis.grid(axis="y", alpha=0.25)
    figure.suptitle("Structured highlight versus energy-matched noise")
    figure.tight_layout(rect=(0, 0, 1, 0.94))
    path = FIGURE_DIR / "feature_shift_boxplot.png"
    figure.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(figure)
    return path


def save_layer_amplification_plot():
    figure, axis = plt.subplots(figsize=(6.8, 4.8))
    x = np.arange(len(FEATURE_LAYERS))
    for condition, column in (
        ("Highlight", "highlight_feature_relative_l2"),
        ("Matched noise", "noise_feature_relative_l2"),
    ):
        medians, q1, q3 = [], [], []
        for layer_name in FEATURE_LAYERS:
            values = PER_IMAGE_TABLE.loc[
                PER_IMAGE_TABLE["layer"] == layer_name, column
            ]
            medians.append(values.median())
            q1.append(values.quantile(0.25))
            q3.append(values.quantile(0.75))
        medians = np.asarray(medians)
        axis.plot(
            x,
            medians,
            marker="o",
            linewidth=2,
            label=condition,
            color=CONDITION_COLORS[condition],
        )
        axis.fill_between(
            x, q1, q3, alpha=0.16, color=CONDITION_COLORS[condition]
        )
    axis.set_xticks(x, FEATURE_LAYERS)
    axis.set_title(f"{MODEL_NAME}: layer-wise propagation")
    axis.set_ylabel("Median relative feature L2 shift")
    axis.grid(axis="y", alpha=0.25)
    axis.legend(frameon=False)
    figure.tight_layout()
    path = FIGURE_DIR / "layer_amplification.png"
    figure.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(figure)
    return path


def save_prediction_consistency_plot():
    deep = PER_IMAGE_TABLE[PER_IMAGE_TABLE["layer"] == FEATURE_LAYERS[-1]]
    values = [
        1.0 - deep["highlight_prediction_dice_to_fmm"],
        1.0 - deep["noise_prediction_dice_to_fmm"],
    ]
    figure, axis = plt.subplots(figsize=(6.2, 4.8))
    boxes = axis.boxplot(
        values,
        labels=["Highlight", "Matched\nnoise"],
        patch_artist=True,
        showfliers=False,
    )
    for box, condition in zip(boxes["boxes"], CONDITION_COLORS):
        box.set_facecolor(CONDITION_COLORS[condition])
        box.set_alpha(0.72)
    axis.set_title(f"{MODEL_NAME}: prediction instability")
    axis.set_ylabel("Prediction inconsistency (1 - Dice)")
    axis.grid(axis="y", alpha=0.25)
    figure.tight_layout()
    path = FIGURE_DIR / "prediction_consistency.png"
    figure.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(figure)
    return path


def save_feature_prediction_correlation():
    deep = PER_IMAGE_TABLE[PER_IMAGE_TABLE["layer"] == FEATURE_LAYERS[-1]]
    figure, axis = plt.subplots(figsize=(7.2, 5.5))
    for dataset_name in TEST_DATASETS:
        selected = deep[deep["dataset"] == dataset_name]
        axis.scatter(
            selected["highlight_feature_relative_l2"],
            selected["highlight_probability_mae"],
            s=16,
            alpha=0.48,
            color=DATASET_COLORS[dataset_name],
            label=dataset_name,
        )
    rho, p_value = safe_spearman(
        deep["highlight_feature_relative_l2"],
        deep["highlight_probability_mae"],
    )
    axis.set_title(
        f"{MODEL_NAME}: feature shift vs prediction change\n"
        f"Spearman rho={rho:.3f}, p={p_value:.2g}"
    )
    axis.set_xlabel("Deep feature relative L2 shift")
    axis.set_ylabel("Prediction probability MAE")
    axis.grid(alpha=0.2)
    axis.legend(frameon=False, fontsize=8)
    figure.tight_layout()
    path = FIGURE_DIR / "feature_prediction_correlation.png"
    figure.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(figure)
    return path


def save_nonlocal_shift_plot():
    figure, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), sharey=True)
    for axis, layer_name in zip(axes, FEATURE_LAYERS):
        group = PER_IMAGE_TABLE[PER_IMAGE_TABLE["layer"] == layer_name]
        values = [
            group["highlight_outside_mask_shift_ratio"],
            group["noise_outside_mask_shift_ratio"],
        ]
        boxes = axis.boxplot(
            values,
            labels=["Highlight", "Matched\nnoise"],
            patch_artist=True,
            showfliers=False,
        )
        for box, condition in zip(boxes["boxes"], CONDITION_COLORS):
            box.set_facecolor(CONDITION_COLORS[condition])
            box.set_alpha(0.72)
        axis.set_ylim(0, 1)
        axis.set_title(f"{MODEL_NAME} | {layer_name}")
        axis.set_ylabel("Feature shift outside dilated mask")
        axis.grid(axis="y", alpha=0.25)
    figure.suptitle("Non-local propagation of localized perturbations")
    figure.tight_layout(rect=(0, 0, 1, 0.94))
    path = FIGURE_DIR / "nonlocal_feature_shift.png"
    figure.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(figure)
    return path


def save_dataset_heatmap():
    deep = PER_IMAGE_TABLE[PER_IMAGE_TABLE["layer"] == FEATURE_LAYERS[-1]]
    values = []
    for dataset_name in TEST_DATASETS:
        selected = deep.loc[
            deep["dataset"] == dataset_name,
            "delta_feature_l2_highlight_minus_noise",
        ]
        values.append(selected.median())
    matrix = np.asarray([values], dtype=np.float64)
    limit = max(float(np.max(np.abs(matrix))), 1e-6)
    figure, axis = plt.subplots(figsize=(10.5, 2.8))
    image = axis.imshow(
        matrix, cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto"
    )
    axis.set_xticks(
        np.arange(len(TEST_DATASETS)), TEST_DATASETS, rotation=18, ha="right"
    )
    axis.set_yticks([0], [MODEL_NAME])
    for column_index, value in enumerate(matrix[0]):
        axis.text(
            column_index, 0, f"{value:+.4f}",
            ha="center", va="center", color="black", fontsize=9,
        )
    axis.set_title("Median deep-feature shift: Highlight minus matched noise")
    figure.colorbar(image, ax=axis, shrink=0.82, label="Delta relative L2")
    figure.tight_layout()
    path = FIGURE_DIR / "dataset_effect_heatmap.png"
    figure.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(figure)
    return path


GLOBAL_FIGURES = [
    save_feature_shift_boxplot(),
    save_layer_amplification_plot(),
    save_prediction_consistency_plot(),
    save_feature_prediction_correlation(),
    save_nonlocal_shift_plot(),
    save_dataset_heatmap(),
]
for figure_path in GLOBAL_FIGURES:
    print("Figure:", figure_path.resolve())


Figure: C:\Users\w\Raw-vs-FMM-Train-Inference-Wilcoxon-PairedT\bad-case\simple_feature_interference_results\FPN+V3\figures\feature_shift_boxplot.png
Figure: C:\Users\w\Raw-vs-FMM-Train-Inference-Wilcoxon-PairedT\bad-case\simple_feature_interference_results\FPN+V3\figures\layer_amplification.png
Figure: C:\Users\w\Raw-vs-FMM-Train-Inference-Wilcoxon-PairedT\bad-case\simple_feature_interference_results\FPN+V3\figures\prediction_consistency.png
Figure: C:\Users\w\Raw-vs-FMM-Train-Inference-Wilcoxon-PairedT\bad-case\simple_feature_interference_results\FPN+V3\figures\feature_prediction_correlation.png
Figure: C:\Users\w\Raw-vs-FMM-Train-Inference-Wilcoxon-PairedT\bad-case\simple_feature_interference_results\FPN+V3\figures\nonlocal_feature_shift.png
Figure: C:\Users\w\Raw-vs-FMM-Train-Inference-Wilcoxon-PairedT\bad-case\simple_feature_interference_results\FPN+V3\figures\dataset_effect_heatmap.png


In [9]:
# ============================================================
# 9. 代表性案例：输入、预测和深层特征变化热图
# ============================================================
def normalized_heatmap(reference_feature, changed_feature):
    difference = feature_difference_map(reference_feature, changed_feature)
    high = float(np.percentile(difference, 99))
    normalized = np.clip(difference / max(high, 1e-12), 0.0, 1.0)
    resized = cv2.resize(normalized, IMAGE_SIZE, interpolation=cv2.INTER_CUBIC)
    colored = plt.get_cmap("inferno")(resized)[..., :3]
    return colored


def confusion_map(probability, ground_truth):
    prediction = probability[..., 0] >= PREDICTION_THRESHOLD
    truth = ground_truth_to_2d(ground_truth) >= PREDICTION_THRESHOLD
    if prediction.shape != truth.shape:
        raise ValueError(
            f"Prediction/GT shape mismatch: {prediction.shape} != {truth.shape}"
        )
    result = np.zeros((IMAGE_HEIGHT, IMAGE_WIDTH, 3), dtype=np.float32)
    result[np.logical_and(prediction, truth)] = (0.10, 0.68, 0.32)  # TP green
    result[np.logical_and(prediction, ~truth)] = (0.90, 0.16, 0.18)  # FP red
    result[np.logical_and(~prediction, truth)] = (0.18, 0.42, 0.88)  # FN blue
    return result


def highlight_overlay(raw_image, highlight_mask):
    overlay = raw_image.copy()
    yellow = np.zeros_like(overlay)
    yellow[..., 0] = 1.0
    yellow[..., 1] = 0.90
    overlay[highlight_mask] = 0.45 * overlay[highlight_mask] + 0.55 * yellow[highlight_mask]
    return overlay


def infer_one_case(feature_model, dataset_name, image_name):
    variants = build_input_variants(dataset_name, image_name)
    inputs = np.asarray([
        variants["fmm"],
        variants["highlight"],
        variants["noises"][0],
    ], dtype=np.float32)
    outputs = feature_model.predict(inputs, batch_size=3, verbose=0)
    return variants, {
        "fmm_features": {
            layer_name: outputs[index][0]
            for index, layer_name in enumerate(FEATURE_LAYERS)
        },
        "highlight_features": {
            layer_name: outputs[index][1]
            for index, layer_name in enumerate(FEATURE_LAYERS)
        },
        "noise_features": {
            layer_name: outputs[index][2]
            for index, layer_name in enumerate(FEATURE_LAYERS)
        },
        "fmm_probability": outputs[-1][0],
        "highlight_probability": outputs[-1][1],
        "noise_probability": outputs[-1][2],
    }


def save_case_figure(model_name, row, variants, inference, output_path):
    deep_layer = FEATURE_LAYERS[-1]
    highlight_heatmap = normalized_heatmap(
        inference["fmm_features"][deep_layer],
        inference["highlight_features"][deep_layer],
    )
    noise_heatmap = normalized_heatmap(
        inference["fmm_features"][deep_layer],
        inference["noise_features"][deep_layer],
    )
    panels = (
        (variants["raw"], "Original Raw image"),
        (variants["fmm"], "99.5% + FMM image"),
        (variants["highlight"], "Isolated highlight condition"),
        (variants["noises"][0], "Energy-matched noise"),
        (highlight_overlay(variants["raw"], variants["highlight_mask"]), "Highlight mask"),
        (ground_truth_to_2d(variants["ground_truth"]), "Ground truth"),
        (
            confusion_map(
                inference["highlight_probability"], variants["ground_truth"]
            ),
            "Highlight prediction error",
        ),
        (
            confusion_map(inference["fmm_probability"], variants["ground_truth"]),
            "FMM prediction error",
        ),
        (highlight_heatmap, "Deep shift: Highlight vs FMM"),
        (noise_heatmap, "Deep shift: Noise vs FMM"),
    )
    figure, axes = plt.subplots(2, 5, figsize=(18, 7.6))
    for axis, (panel, title) in zip(axes.ravel(), panels):
        axis.imshow(panel, cmap="gray" if panel.ndim == 2 else None)
        axis.set_title(title, fontsize=10)
        axis.axis("off")
    figure.suptitle(
        f"{model_name} | {row.dataset} | {row.image_name}\n"
        f"Deep feature L2: highlight={row.highlight_feature_relative_l2:.4f}, "
        f"noise={row.noise_feature_relative_l2:.4f}; "
        f"prediction MAE={row.highlight_probability_mae:.4f}",
        fontsize=12,
    )
    figure.tight_layout(rect=(0, 0, 1, 0.93))
    figure.savefig(output_path, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(figure)
    return {
        "raw": variants["raw"],
        "fmm": variants["fmm"],
        "mask": highlight_overlay(variants["raw"], variants["highlight_mask"]),
        "highlight_heatmap": highlight_heatmap,
        "prediction_error": confusion_map(
            inference["highlight_probability"], variants["ground_truth"]
        ),
    }


deep_table = PER_IMAGE_TABLE[
    PER_IMAGE_TABLE["layer"] == FEATURE_LAYERS[-1]
].copy()
CASE_ROWS = (
    deep_table.sort_values(
        ["model", "delta_feature_l2_highlight_minus_noise"],
        ascending=[True, False],
    )
    .groupby("model", sort=False, group_keys=False)
    .head(CASE_TOP_K)
    .reset_index(drop=True)
)

case_outputs = []
overview_panels = []
for model_name in MODEL_CONFIGS:
    print("Rendering cases for", model_name)
    feature_model = build_feature_model(model_name)
    model_rows = CASE_ROWS[CASE_ROWS["model"] == model_name]
    for case_index, row in enumerate(model_rows.itertuples(index=False)):
        variants, inference = infer_one_case(
            feature_model, row.dataset, row.image_name
        )
        safe_name = re.sub(r"[^A-Za-z0-9._+-]+", "_", Path(row.image_name).stem)
        output_path = CASE_FIGURE_DIR / (
            f"{model_name.replace('+', '_')}__{row.dataset}__{safe_name}.png"
        )
        overview = save_case_figure(
            model_name, row, variants, inference, output_path
        )
        case_outputs.append({
            "model": model_name,
            "dataset": row.dataset,
            "image_name": row.image_name,
            "deep_highlight_feature_l2": row.highlight_feature_relative_l2,
            "deep_noise_feature_l2": row.noise_feature_relative_l2,
            "delta_highlight_minus_noise": row.delta_feature_l2_highlight_minus_noise,
            "figure_path": str(output_path.resolve()),
        })
        if case_index == 0:
            overview_panels.append((model_name, row, overview))
    del feature_model
    tf.keras.backend.clear_session()
    gc.collect()

CASE_TABLE = pd.DataFrame(case_outputs)
CASE_TABLE.to_csv(
    RESULT_DIR / "representative_cases.csv", index=False, encoding="utf-8-sig"
)

overview_path = FIGURE_DIR / "representative_overview.png"
figure, axes = plt.subplots(len(overview_panels), 5, figsize=(16, 3.6 * len(overview_panels)))
if len(overview_panels) == 1:
    axes = np.asarray([axes])
for row_index, (model_name, row, panels) in enumerate(overview_panels):
    items = (
        (panels["raw"], "Raw"),
        (panels["fmm"], "FMM"),
        (panels["mask"], "Highlight mask"),
        (panels["highlight_heatmap"], "Deep feature shift"),
        (panels["prediction_error"], "Prediction error"),
    )
    for axis, (panel, title) in zip(axes[row_index], items):
        axis.imshow(panel)
        axis.set_title(title)
        axis.axis("off")
    axes[row_index, 0].set_ylabel(
        f"{model_name}\n{row.dataset}\n{row.image_name}", fontsize=9
    )
figure.suptitle(
    "Representative structured-highlight interference cases\n"
    "Feature heatmap shows mixed10 absolute activation change",
    fontsize=13,
)
figure.tight_layout(rect=(0, 0, 1, 0.92))
figure.savefig(overview_path, dpi=FIGURE_DPI, bbox_inches="tight")
plt.close(figure)
print("Representative overview:", overview_path.resolve())


Rendering cases for FPN+V3
FPN+V3 Raw weight: D:\WJY\paper1\best weight\2.different mode\FPN+V3\img\FPN+V3_img_bestweight.h5
FPN+V3 outputs: [(None, None, None, 256), (None, None, None, 2048), (None, None, None, 1)]
Representative overview: C:\Users\w\Raw-vs-FMM-Train-Inference-Wilcoxon-PairedT\bad-case\simple_feature_interference_results\FPN+V3\figures\representative_overview.png


In [10]:
# ============================================================
# 10. 最终一致性检查与结果解释提示
# ============================================================
expected_layers = set(FEATURE_LAYERS)
if set(PER_IMAGE_TABLE["layer"].unique()) != expected_layers:
    raise AssertionError("Missing feature layers in per-image table")
if set(PER_IMAGE_TABLE["model"].unique()) != set(MODEL_CONFIGS):
    raise AssertionError("Missing model results")
numeric = PER_IMAGE_TABLE.select_dtypes(include=[np.number]).to_numpy()
if not np.isfinite(numeric).all():
    raise FloatingPointError("NaN or Inf found in final per-image table")
if not np.allclose(
    PER_IMAGE_TABLE["noise_energy_ratio"].to_numpy(),
    1.0,
    atol=NOISE_ENERGY_TOLERANCE,
):
    raise AssertionError("Matched-noise energy validation failed")

required_files = [
    PER_IMAGE_CSV,
    STATISTICS_CSV,
    SUMMARY_XLSX,
    overview_path,
    *GLOBAL_FIGURES,
]
missing = [str(path) for path in required_files if not Path(path).is_file()]
if missing:
    raise FileNotFoundError("Missing output files:\n" + "\n".join(missing))

all_summary = SUMMARY_TABLE[SUMMARY_TABLE["dataset"] == "ALL"].copy()
display(all_summary[[
    "model",
    "layer",
    "N",
    "highlight_feature_l2_median",
    "noise_feature_l2_median",
    "median_delta_highlight_minus_noise",
    "highlight_greater_than_noise_rate",
    "paired_rank_biserial",
    "wilcoxon_p",
    "rho_feature_shift_vs_prediction_mae",
    "rho_p_value",
]].round(5))

print("\nAll checks passed.")
print("结果解释原则：")
print("1. Highlight 特征偏移显著大于 matched noise，才支持结构化干扰。")
print("2. mixed10 的差异比 mixed0 更明显，才支持深层传播或放大。")
print("3. 特征偏移与预测 MAE 正相关，才支持表征变化具有任务相关性。")
print("4. 如果上述方向不一致，应将论文表述改为 may/can，而不能声称已证明。")
print("Results:", RESULT_DIR.resolve())


,model,layer,N,highlight_feature_l2_median,noise_feature_l2_median,median_delta_highlight_minus_noise,highlight_greater_than_noise_rate,paired_rank_biserial,wilcoxon_p,rho_feature_shift_vs_prediction_mae,rho_p_value
5,FPN+V3,mixed0,796,0.05466,0.05238,0.00166,0.76508,0.59592,0.0,0.42282,0.0
11,FPN+V3,mixed10,796,0.03979,0.03629,0.00209,0.69095,0.48711,0.0,0.67567,0.0



All checks passed.
结果解释原则：
1. Highlight 特征偏移显著大于 matched noise，才支持结构化干扰。
2. mixed10 的差异比 mixed0 更明显，才支持深层传播或放大。
3. 特征偏移与预测 MAE 正相关，才支持表征变化具有任务相关性。
4. 如果上述方向不一致，应将论文表述改为 may/can，而不能声称已证明。
Results: C:\Users\w\Raw-vs-FMM-Train-Inference-Wilcoxon-PairedT\bad-case\simple_feature_interference_results\FPN+V3
